In [16]:
# Cell 1: Imports and Setup
import pandas as pd
from Bio import SeqIO
from collections import Counter
import os
import time

# You may need to adjust these paths if your notebook is not in the project root!
DATA_DIR = "../data/Train"  # Check your relative path!
TEST_FASTA_PATH = "../data/Test/testsuperset.fasta"
OUTPUT_DIR = "./submissions"
NUM_PREDICTIONS = 10
# ... (rest of the code) ...

In [9]:
# LOAD AND ANALYSE TRAINING LABELS
def get_terms_frequencies(file_path):
    """Load the ground truth and counts the frequencies of each GO term."""
    print("1. Loading training terms...")

    # The file is TSV (tab separated) and has columns: EntryID, term, aspect
    # only care about the 'term' column for naive baseline
    train_terms = pd.read_csv(
        file_path,
        sep='\t',
        usecols=['EntryID', 'term'],
        header=0 # first row is header
    )

    # use counter for fast freq calc
    all_terms = train_terms['term'].tolist()
    term_counts = Counter(all_terms)

    # Get the most common terms and their freq/prob
    # Use the freq as the 'Score' (confidence)
    most_common = term_counts.most_common(NUM_PREDICTIONS)

    # Convert to a DataFrame for easier handling
    term_freq_df = pd.DataFrame(most_common, columns=['go_term', 'frequency'])
    total_annotations = term_freq_df['frequency'].sum()
    term_freq_df['score'] = term_freq_df['frequency'] / total_annotations

    print(f" -> Total unique GO terms: {len(term_counts):,}")
    print(f" -> using the top {NUM_PREDICTIONS} terms for predictions.")
    return term_freq_df.drop(columns=['frequency'])

In [10]:
# Cell 2: Run the Frequency Calculation
TRAIN_TERMS_PATH = os.path.join(DATA_DIR, "train_terms.tsv")
naive_scores_df = get_terms_frequencies(TRAIN_TERMS_PATH)

# Check the result
print(naive_scores_df.head())
print(f"Total rows in score table: {len(naive_scores_df)}") 

# Expected output structure:
#        go_term     score
# 0  GO:0005515  0.009876  <- (e.g., 'protein binding')
# 1  GO:0006412  0.008543  <- (e.g., 'translation')
# ...

1. Loading training terms...
 -> Total unique GO terms: 26,125
 -> using the top 10 terms for predictions.
      go_term     score
0  GO:0005515  0.334285
1  GO:0005634  0.131709
2  GO:0005829  0.129300
3  GO:0005886  0.100644
4  GO:0005737  0.093623
Total rows in score table: 10


In [17]:
# load test seq (testsuperset.fasta)
def get_test_proteins(file_path):
    """Parses the FASTA file to get a list of protein IDs."""
    print("2. Loading test proteins...")
    test_proteins = []
    
    # Biopython's SeqIO for FASTA parsing
    for record in SeqIO.parse(file_path, "fasta"):
        # "The ID in FASTA is formatted like "sp|ID|NAME"
        # We need he unique ID part
        parts = record.id.split('|')
        protein_id = parts[1] if len(parts) > 1 else record.id
        test_proteins.append(protein_id)
    
    print(f" -> total test proteins: {len(test_proteins):,}")
    return test_proteins


In [18]:
test_ids = get_test_proteins(TEST_FASTA_PATH)

2. Loading test proteins...
 -> total test proteins: 224,309


In [13]:
# --- 3. Generate Submission File (The OPTIMIZED Naive Prediction) ---
def generate_naive_submission(test_proteins, term_scores):
    """
    Predicts the same set of high-frequency terms for every test protein 
    using vectorized Pandas operations for speed.
    """
    print("3. Generating naive submission (Optimized)...")
    
    # --- The Vectorized Fix ---
    
    # 1. Create a Series for the Protein IDs
    # This Series has the IDs repeated 1500 times each (224k * 1500 total)
    protein_series = pd.Series(test_proteins)
    
    # 2. Use the 'product' logic: repeat the GO Terms for the entire length
    # of the expanded protein_series, then repeat the protein list 1500 times.
    # A faster way is using the built-in pd.MultiIndex.from_product
    
    # Use Pandas product to create the Cartesian product (all combinations)
    multi_index = pd.MultiIndex.from_product(
        [test_proteins, term_scores['go_term']], 
        names=['EntryID', 'go_term']
    )
    
    # Convert the MultiIndex to a DataFrame
    submission_df = multi_index.to_frame(index=False)
    
    # 3. Merge the scores onto the new DataFrame
    # This is a fast, vectorized join based on the 'term' column.
    submission_df = submission_df.merge(term_scores, on='go_term', how='left')
    
    # Final cleanup and formatting
    submission_df = submission_df[['EntryID', 'go_term', 'score']]
    submission_df['score'] = submission_df['score'].apply(lambda x: f"{x:.6f}") # Format score
    
    # --- End of Vectorized Fix ---

    # Save the submission file
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    output_file = os.path.join(OUTPUT_DIR, f"naive_baseline_optimized_{timestamp}.tsv")

    # Submission must be TSV, no index, no header
    # NOTE: Writing 336M rows will still take time, but creating the DF is fast.
    submission_df.to_csv(output_file, sep='\t', index=False, header=False)
    
    print(f"4. Success! Submission file saved to: {output_file}")
    print(f"   -> Total prediction rows: {len(submission_df):,}")

    return submission_df

In [14]:
generate_naive_submission(test_ids, naive_scores_df)

3. Generating naive submission (Optimized)...
4. Success! Submission file saved to: ./submissions/naive_baseline_optimized_20251204-190932.tsv
   -> Total prediction rows: 2,243,090


,EntryID,go_term,score
0,A0A0C5B5G6,GO:0005515,0.334285
1,A0A0C5B5G6,GO:0005634,0.131709
2,A0A0C5B5G6,GO:0005829,0.129300
3,A0A0C5B5G6,GO:0005886,0.100644
4,A0A0C5B5G6,GO:0005737,0.093623
...,...,...,...
2243085,Q9ZB81,GO:0005739,0.057580
2243086,Q9ZB81,GO:0005654,0.050223
2243087,Q9ZB81,GO:0016020,0.035329
2243088,Q9ZB81,GO:0042802,0.035171


In [19]:
# Cell 1: Function to calculate Amino Acid Composition (AAC)
def calculate_aac_features(fasta_path):
    """
    Reads the test FASTA file and calculates the frequency of the 20 standard 
    amino acids (A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y) 
    for every protein.
    """
    print("5. Calculating Amino Acid Composition features...")
    
    protein_features = {}
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY' # 20 standard amino acids
    
    # 1. Parse the FASTA file again
    for record in SeqIO.parse(fasta_path, "fasta"):
        
        # Extract the same protein ID as before
        parts = record.id.split('|')
        protein_id = parts[1] if len(parts) > 1 else record.id
        
        sequence = str(record.seq).upper().replace('*', '').replace('X', '') # Clean the sequence
        seq_length = len(sequence)
        
        if seq_length == 0:
            # Skip empty sequences
            continue
            
        # 2. Count frequencies
        counts = Counter(sequence)
        
        # 3. Store the frequencies (features)
        features = {}
        for aa in amino_acids:
            # Store normalized frequency (count / length)
            features[aa] = counts[aa] / seq_length
        
        protein_features[protein_id] = features

    # 4. Convert to DataFrame
    aac_df = pd.DataFrame.from_dict(protein_features, orient='index')
    aac_df.index.name = 'EntryID'
    
    print(f" -> AAC features calculated for {len(aac_df):,} proteins.")
    return aac_df

In [20]:
# Cell 2: Execute AAC calculation
aac_features_df = calculate_aac_features(TEST_FASTA_PATH)

# Review the resulting feature table (should have 20 columns)
print("\nAAC Features Head:")
print(aac_features_df.head())
print(f"\nDataFrame shape: {aac_features_df.shape}")

5. Calculating Amino Acid Composition features...
 -> AAC features calculated for 224,309 proteins.

AAC Features Head:
                   A         C         D         E         F         G  \
EntryID                                                                  
A0A0C5B5G6  0.000000  0.000000  0.000000  0.062500  0.062500  0.062500   
A0A1B0GTW7  0.068528  0.036802  0.034264  0.044416  0.024112  0.090102   
A0JNW5      0.049863  0.017077  0.060109  0.066940  0.035519  0.037568   
A0JP26      0.053356  0.037866  0.063683  0.092943  0.015491  0.056799   
A0PK11      0.107759  0.021552  0.021552  0.051724  0.064655  0.073276   

                   H         I         K         L         M         N  \
EntryID                                                                  
A0A0C5B5G6  0.000000  0.062500  0.062500  0.062500  0.125000  0.000000   
A0A1B0GTW7  0.038071  0.021574  0.032995  0.143401  0.016497  0.017766   
A0JNW5      0.032787  0.057377  0.071038  0.099044  0.023907  0.0